# LMOT / EST / Sinkhorn — Simulation

Notebook gọi code chung trong repo, không chứa bản sao của thuật toán.

**Chuẩn bị:** upload và giải nén archive để có folder `/content/lmot-paper`, hoặc clone repo GitHub của bạn vào Colab. Sau đó chỉnh `REPO_DIR` ở cell dưới nếu tên folder khác.

Bắt đầu bằng `smoke.yaml`; sau đó chọn `pilot.yaml`, `weights.yaml` hoặc `scaling.yaml`. Smoke là kiểm tra pipeline, chưa phải kết quả paper-scale.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess

REPO_DIR = Path('/content/lmot-paper')  # Chỉnh theo folder repo của bạn.
if not (REPO_DIR / 'pyproject.toml').exists():
    raise FileNotFoundError('Hãy upload/clone repo rồi chỉnh REPO_DIR trước.')
os.chdir(REPO_DIR)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
# Editable installs made during an existing kernel may need the src path now.
if str(REPO_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_DIR / 'src'))


## 1. Chọn config
Sửa YAML nếu cần thay N, overlap, L, epsilon, seeds hoặc số lần đo. Các method dùng cùng input; LMOT và EST dùng đúng cùng projections. Sinkhorn được đo một lần cho mỗi dataset/epsilon.


In [ ]:
CONFIG = REPO_DIR / 'experiments/simulation/configs/smoke.yaml'
print(CONFIG.read_text())


In [ ]:
from experiments.simulation.run import run
run_dir = run(CONFIG)


## 2. Bảng kết quả
`summary.tsv` có thể tải xuống để mở bằng Google Sheets. Kiểm tra cột `status`: `not_converged` và `skipped_size` không phải kết quả hội tụ. Memory trong CSV là traced allocations, không phải toàn bộ RSS.


In [ ]:
import csv
columns = ['n', 'd', 'sweep_value', 'method', 'projection_kind', 'L',
           'epsilon', 'median_ms', 'cost', 'status']
with (run_dir / 'summary.csv').open() as f:
    rows = list(csv.DictReader(f))
print('\t'.join(columns))
for row in rows:
    print('\t'.join(str(row.get(column, '')) for column in columns))
print('\nFile để tải:', run_dir / 'summary.tsv')


In [ ]:
from IPython.display import display, Image
for path in sorted((run_dir / 'figures').glob('*.png')):
    display(Image(filename=str(path)))


## 3. Kiểm tra correctness (tùy chọn)
Reference dense trong tests độc lập với implementation tối ưu. Sinkhorn mặc định dùng POT log-domain; test cũng đối chiếu với backend SciPy độc lập.


In [ ]:
subprocess.check_call([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'])
